# 02 — Graph Construction

**Rift: Graph ML for Fraud Detection, Replay, and Audit**

This notebook builds a heterogeneous transaction graph with 5 node types and 7 edge types, then explores its structure and properties.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AngelP17/Rift/blob/main/notebooks/02_graph_construction.ipynb)


## Environment Setup


In [ ]:
# Clone repo and install dependencies (run once)
import subprocess, sys, os

REPO = "https://github.com/AngelP17/Rift.git"
if not os.path.exists("/content/Rift"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/Rift"], check=True)

os.chdir("/content/Rift")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "polars", "numpy", "pandas", "scikit-learn", "xgboost", "duckdb",
    "shap", "structlog", "python-dotenv", "rich", "jinja2", "pyarrow"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "torch", "--index-url", "https://download.pytorch.org/whl/cpu"], check=False)
sys.path.insert(0, "src")

try:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    device = "cpu"
print(f"Device: {device}")


## Generate Data and Build Features


In [ ]:
from data.generator import generate_transactions
from features.engine import build_features, FEATURE_COLUMNS

df = generate_transactions(n_txns=20_000, n_users=1_000, n_merchants=300, fraud_rate=0.03, seed=42)
df_feat = build_features(df)
print(f"Transactions: {len(df_feat):,}")
print(f"Feature columns: {len(FEATURE_COLUMNS)}")


## Build Heterogeneous Graph

The graph connects 5 node types through 7 edge types:
- **user** → initiates → **transaction**
- **transaction** → at → **merchant**
- **transaction** → via → **device**
- **transaction** → from → **account**
- **user** → uses → **device**
- **user** → shops_at → **merchant**
- **account** → linked → **device**


In [ ]:
from graph.builder import build_graph

g = build_graph(df_feat, FEATURE_COLUMNS)

print("=== Node Types ===")
for ntype, count in g.node_types.items():
    print(f"  {ntype}: {count:,} nodes")

print(f"\n=== Edge Types ===")
for etype, ei in g.edge_index.items():
    src_type, rel, dst_type = etype
    print(f"  ({src_type}) --[{rel}]--> ({dst_type}): {ei.shape[1]:,} edges")

print(f"\nTotal nodes: {sum(g.node_types.values()):,}")
print(f"Total edges: {sum(ei.shape[1] for ei in g.edge_index.values()):,}")


## Homogeneous Projection

For GNN training, we project the heterogeneous graph into a homogeneous transaction graph where two transactions share an edge if they have a common neighbor (same user, device, or merchant).


In [ ]:
from graph.hetero_graph import to_homogeneous_projection

x, edge_index, labels = to_homogeneous_projection(g, "transaction")
print(f"Transaction features: {x.shape}")
print(f"Projected edges: {edge_index.shape[1]:,}")
print(f"Labels: {labels.shape} (fraud: {labels.sum()}, legit: {(labels == 0).sum()})")


## Graph Motif Features


In [ ]:
from graph.motifs import compute_degree_features, compute_motif_features

degrees = compute_degree_features(g, "transaction")
print(f"Degree features shape: {degrees.shape}")
print(f"Mean in-degree: {degrees[:, 0].mean():.2f}")
print(f"Mean out-degree: {degrees[:, 1].mean():.2f}")

motifs = compute_motif_features(g, "transaction")
print(f"\nMotif features shape: {motifs.shape}")
print(f"Mean triangle count: {motifs[:, 2].mean():.2f}")


## Windowed Graphs

Rift supports rolling-window graph construction for temporal evaluation.


In [ ]:
from graph.windows import build_rolling_graphs

graphs = build_rolling_graphs(df_feat, window_days=30, step_days=30, feature_cols=FEATURE_COLUMNS)
print(f"Rolling windows: {len(graphs)}")
for ts, wg in graphs[:3]:
    print(f"  Window ending {ts}: {wg.node_types.get('transaction', 0):,} txns")


## Graph Statistics Summary


In [ ]:
import numpy as np

print("=== Fraud Graph Connectivity ===")
fraud_mask = labels.numpy() == 1
legit_mask = labels.numpy() == 0

fraud_deg = degrees[fraud_mask]
legit_deg = degrees[legit_mask]

print(f"Fraud txns - avg in-degree: {fraud_deg[:, 0].mean():.2f}, avg out-degree: {fraud_deg[:, 1].mean():.2f}")
print(f"Legit txns - avg in-degree: {legit_deg[:, 0].mean():.2f}, avg out-degree: {legit_deg[:, 1].mean():.2f}")

print(f"\nGraph density: {edge_index.shape[1] / (x.shape[0] ** 2):.6f}")
